### Assignment for 2.3 Data Encoding and Data Flow

#### Question 1: Implement a simple Thrift server and client that defines a `Student` struct with fields `name` (string), `age` (integer), and `courses` (list of strings). Include a service `School` with a method `enrollCourse` that takes a `Student` record and a course name, adds the course to the student's course list, and returns the updated `Student` record.

###### student.thrift

In [3]:
%%writefile ../sandbox/student.thrift
struct Student{
    1: required string name,
    2: required i64 age,
    3: required list<string> courses
}

service School {
    Student enrollCourse(1: required Student student, 2: required string course)
}

Overwriting ../sandbox/student.thrift


##### Thrift server (student_server.py)

In [4]:
%%writefile ../sandbox/student_server.py
import thriftpy2
student_thrift = thriftpy2.load("../sandbox/student.thrift", module_name="student_thrift")

from thriftpy2.rpc import make_server

class School(object):
    def enrollCourse(self, student, course):
        student.courses.append(course)
        return student

server = make_server(student_thrift.School, School(), client_timeout=None)
server.serve()

Overwriting ../sandbox/student_server.py


##### Thrift server (student_client.py)

In [8]:
import thriftpy2
student_thrift = thriftpy2.load("../sandbox/student.thrift", module_name="student_thrift")

from thriftpy2.rpc import make_client

school = make_client(student_thrift.School, timeout=None)   

# Create a Student instance and enroll in a course
student = student_thrift.Student(name="Alice", age=20, courses=["Math", "Science"])
enrolled_student = school.enrollCourse(student, "History")      
print(f"Student Name: {enrolled_student.name}")
print(f"Student Age: {enrolled_student.age}") 
print(f"Enrolled Courses: {', '.join(enrolled_student.courses)}")

Student Name: Alice
Student Age: 20
Enrolled Courses: Math, Science, History


#### Question 2: Implement a simple Protocol Buffers server and client that defines a `Book` message with fields `title` (string), `author` (string), and `page_count` (integer). Include a service `Library` with a method `checkoutBook` that takes a `Book` message and returns the same `Book` message.

##### Protobuf schema (book.proto)

In [12]:
%%writefile ../sandbox/book.proto
syntax = "proto3";

message Book {
  string title= 1;
  string author = 2;
  int32 page_count = 3;
}

service Library {
  rpc checkoutBook(Book) returns (Book) {}
}

Overwriting ../sandbox/book.proto


Run the following command in a terminal to generate the Python code:

```bash
python -m grpc_tools.protoc -I./sandbox --python_out=./sandbox --grpc_python_out=./sandbox ./sandbox/book.proto
```

This will generate the following files:

```bash
book_pb2.py
book_pb2_grpc.py
```

#### Protobuf server (book_server.py)

In [17]:
%%writefile ../sandbox/book_server.py
from concurrent import futures
import grpc
import book_pb2  # Import the generated protobuf code
import book_pb2_grpc # Import the generated gRPC code


class LibraryServicer(book_pb2_grpc.LibraryServicer):
  def checkoutBook(self, request, context):
    return request

server = grpc.server(futures.ThreadPoolExecutor(max_workers=2))
book_pb2_grpc.add_LibraryServicer_to_server(LibraryServicer(), server)
server.add_insecure_port('[::]:50051')
server.start()
server.wait_for_termination()

Writing ../sandbox/book_server.py


 Run `python sandbox/book_server.py` in a new terminal to start the server.

#### Protobuf client (book_client.py)

In [18]:
import sys
sys.path.append('..')
import grpc
import book_pb2
import book_pb2_grpc

with grpc.insecure_channel('localhost:50051') as channel:
    stub = book_pb2_grpc.LibraryStub(channel)
    book = book_pb2.Book(title="A Tale of Two Cities", author="Charles Dickens", page_count=544)
    response = stub.checkoutBook(book)
    print(f"Received book: {response.title} by {response.author}, {response.page_count} pages")

Received book: A Tale of Two Cities by Charles Dickens, 544 pages
